<a href="https://colab.research.google.com/github/AliefReefandi/Practical-Statistics-for-Data-Scientists/blob/main/Chapter_3_Statistical_Experiments_and_Significance_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 3 — Statistical Experiments and Significance Testing

Chapter 3 membahas bagaimana melakukan eksperimen dan pengujian statistik untuk mengetahui apakah suatu perbedaan pada data benar-benar menunjukkan adanya efek atau hanya terjadi karena variasi acak.

# 1. Tujuan Pembelajaran

Setelah mempelajari chapter ini, dapat dipahami:

- A/B Testing
- Hypothesis Testing
- Permutation Test
- p-Value dan Statistical Significance
- t-Test
- Multiple Testing
- ANOVA
- Chi-Square Test
- Multi-Arm Bandit
- Power dan Sample Size

# 2. Ringkasan Chapter

Secara sederhana, alur analisis pada Chapter 3 adalah:

**Data → Eksperimen → Hipotesis → Test Statistic → Statistical Test → p-Value → Interpretasi**

Chapter ini berfokus pada bagaimana membandingkan kelompok atau treatment dan menentukan apakah perbedaan yang ditemukan dapat dijelaskan oleh variasi acak.

# 3. Konsep Teoritis

## 3.1 A/B Testing

A/B Testing digunakan untuk membandingkan dua treatment atau kondisi.

Contoh:

- Page A → desain website lama
- Page B → desain website baru

Tujuannya bukan hanya melihat kelompok mana yang memiliki rata-rata lebih tinggi, tetapi juga mengetahui apakah perbedaan tersebut cukup besar dibandingkan variasi yang mungkin terjadi secara acak.

### Analisis

A/B Testing dapat digunakan untuk mengevaluasi perubahan pada website, produk, atau sistem dengan membandingkan hasil dari dua kelompok.

## 3.2 Hypothesis Testing

Hypothesis Testing digunakan untuk menguji suatu dugaan berdasarkan data.

Terdapat dua hipotesis utama:

- **Null Hypothesis (H₀):** tidak terdapat perbedaan atau efek.
- **Alternative Hypothesis (H₁):** terdapat perbedaan atau efek.

Contoh:

> H₀: Rata-rata waktu kunjungan Page A dan Page B sama.

> H₁: Rata-rata waktu kunjungan Page A dan Page B berbeda.

### Analisis

Pengujian dilakukan untuk mengetahui apakah hasil yang diamati masih dapat dijelaskan oleh variasi acak atau memberikan bukti terhadap adanya perbedaan.

## 3.3 Permutation Test

Permutation Test merupakan metode resampling dengan cara menggabungkan data kemudian mengacak kembali data tersebut untuk melihat seberapa sering perbedaan seperti yang diamati dapat terjadi secara random.

### Analisis

Jika perbedaan yang diamati jarang muncul dalam hasil permutation, maka perbedaan tersebut semakin sulit dijelaskan hanya oleh variasi acak.

Permutation Test memiliki kelebihan karena dapat digunakan dengan relatif sedikit asumsi mengenai distribusi data.

In [1]:
import numpy as np

page_a = np.array([126, 135, 120, 140, 130])
page_b = np.array([160, 155, 170, 165, 162])

observed_diff = page_b.mean() - page_a.mean()

combined = np.concatenate([page_a, page_b])

perm_diffs = []

for _ in range(1000):
    np.random.shuffle(combined)

    new_a = combined[:len(page_a)]
    new_b = combined[len(page_a):]

    perm_diffs.append(new_b.mean() - new_a.mean())

p_value = np.mean(
    np.abs(perm_diffs) >= abs(observed_diff)
)

print("Observed difference:", observed_diff)
print("p-value:", p_value)

Observed difference: 32.20000000000002
p-value: 0.006


## 3.4 p-Value dan Statistical Significance

**p-value** menunjukkan seberapa ekstrem hasil yang diamati jika Null Hypothesis dianggap benar.

Secara sederhana:

- **p-value kecil** → hasil lebih sulit dijelaskan hanya dengan random variation.
- **p-value besar** → hasil masih cukup mungkin terjadi karena random variation.

Nilai signifikansi yang sering digunakan adalah:

**α = 0.05**

### Analisis

p-value membantu dalam mengevaluasi hasil hypothesis testing, tetapi tidak dapat diartikan sebagai probabilitas bahwa Null Hypothesis benar.

## 3.5 t-Test

t-Test digunakan untuk membandingkan rata-rata dari dua kelompok dengan data numerik.

Dalam Python, pengujian dapat dilakukan menggunakan `scipy.stats.ttest_ind`.

### Analisis

t-Test memberikan pendekatan statistik untuk mengetahui apakah terdapat perbedaan rata-rata antara dua kelompok.

In [2]:
from scipy import stats

result = stats.ttest_ind(
    page_a,
    page_b,
    equal_var=False
)

print("t-statistic:", result.statistic)
print("p-value:", result.pvalue)

t-statistic: -7.527145773616173
p-value: 0.00011049007090522815


## 3.6 Multiple Testing

Multiple Testing terjadi ketika banyak pengujian statistik dilakukan secara bersamaan.

Semakin banyak pengujian dilakukan, semakin besar kemungkinan mendapatkan hasil yang terlihat signifikan hanya karena kebetulan.

### Analisis

Ketika terdapat banyak kelompok, melakukan banyak perbandingan satu per satu dapat meningkatkan risiko kesimpulan yang keliru.

Karena itu, metode seperti **ANOVA** dapat digunakan untuk melakukan pengujian secara keseluruhan.

## 3.7 ANOVA

**ANOVA (Analysis of Variance)** digunakan untuk menguji perbedaan antara tiga kelompok atau lebih.

ANOVA menggunakan **F-statistic**, yaitu perbandingan variasi antar kelompok dengan variasi error.

### Analisis

ANOVA membantu menjawab pertanyaan:

> Apakah terdapat perbedaan secara keseluruhan di antara beberapa kelompok?

Dengan demikian, ANOVA dapat digunakan sebelum melakukan analisis perbandingan yang lebih spesifik antar kelompok.

In [3]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

data = pd.DataFrame({
    'Time': [
        164, 172, 177, 156, 195,
        178, 191, 182, 185, 177,
        175, 193, 171, 163, 176,
        155, 166, 164, 170, 168
    ],
    'Page': (
        ['Page 1'] * 5 +
        ['Page 2'] * 5 +
        ['Page 3'] * 5 +
        ['Page 4'] * 5
    )
})

model = smf.ols('Time ~ Page', data=data).fit()

anova_table = sm.stats.anova_lm(model)

print(anova_table)

            df  sum_sq     mean_sq         F    PR(>F)
Page       3.0   831.4  277.133333  2.739825  0.077586
Residual  16.0  1618.4  101.150000       NaN       NaN


## 3.8 Chi-Square Test

Chi-Square Test digunakan terutama untuk data berupa **count, frequency, atau data kategorikal**.

Contohnya adalah untuk melihat apakah dua variabel kategorikal memiliki hubungan atau apakah pola frekuensi yang diamati sesuai dengan asumsi tertentu.

### Analisis

Chi-Square Test dapat digunakan ketika data yang dianalisis berbentuk jumlah atau frekuensi, bukan nilai numerik kontinu.

In [4]:
from scipy.stats import chi2_contingency

table = [
    [30, 20],
    [15, 35]
]

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p)

Chi-square: 7.919191919191919
p-value: 0.004891311452359333


## 3.9 Multi-Arm Bandit

Multi-Arm Bandit merupakan pendekatan untuk eksperimen dengan lebih dari dua treatment.

Berbeda dengan A/B Testing tradisional, Multi-Arm Bandit dapat menggunakan informasi dari hasil eksperimen untuk mengurangi pemberian treatment yang performanya lebih rendah dan lebih banyak mencoba treatment yang memberikan hasil lebih baik.

### Analisis

Pendekatan ini berguna pada eksperimen dengan beberapa pilihan ketika sistem perlu melakukan eksplorasi sekaligus memanfaatkan hasil yang sudah diperoleh.

In [5]:
import numpy as np

results = {
    "A": np.random.binomial(1, 0.10, 100),
    "B": np.random.binomial(1, 0.15, 100),
    "C": np.random.binomial(1, 0.12, 100)
}

for name, data in results.items():
    print(name, data.mean())

A 0.08
B 0.13
C 0.09


## 3.10 Power dan Sample Size

**Effect Size** adalah ukuran perbedaan yang ingin dideteksi.

**Power** adalah kemampuan suatu eksperimen untuk mendeteksi effect size tertentu.

Secara sederhana:

- Effect size semakin kecil → biasanya membutuhkan sample lebih banyak.
- Effect size semakin besar → biasanya lebih mudah dideteksi.
- Sample size yang lebih besar dapat membantu meningkatkan kemampuan mendeteksi suatu efek.

### Analisis

Penentuan sample size penting agar eksperimen tidak menghasilkan kesimpulan yang tidak meyakinkan hanya karena jumlah data yang terlalu sedikit.

# 4. Summary dan Kesimpulan

## Summary

Chapter 3 membahas bagaimana melakukan eksperimen dan pengujian statistik untuk mengetahui apakah suatu perbedaan pada data dapat dijelaskan oleh variasi acak atau menunjukkan adanya efek.

Konsep utama yang dipelajari adalah:

- A/B Testing
- Hypothesis Testing
- Permutation Test
- p-Value
- t-Test
- ANOVA
- Chi-Square Test
- Multi-Arm Bandit
- Power dan Sample Size

## Kesimpulan

Dari Chapter 3 dapat dipahami bahwa perbedaan pada data tidak selalu berarti terdapat efek yang nyata. Pengujian statistik diperlukan untuk mempertimbangkan kemungkinan variasi acak.

Pemilihan metode pengujian perlu disesuaikan dengan jenis data, jumlah kelompok, tujuan eksperimen, dan ukuran sample.